In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

In [2]:
df  = pd.read_csv('owid-covid-data.csv') 
df.head()

,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index,population,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million
0,AFG,Asia,Afghanistan,2020-01-05,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
1,AFG,Asia,Afghanistan,2020-01-06,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
2,AFG,Asia,Afghanistan,2020-01-07,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
3,AFG,Asia,Afghanistan,2020-01-08,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
4,AFG,Asia,Afghanistan,2020-01-09,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN


In [3]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
total_cases,411804.0,7.365292e+06,4.477582e+07,0.00,6280.750,63653.00,7.582720e+05,7.758668e+08
new_cases,410159.0,8.017360e+03,2.296649e+05,0.00,0.000,0.00,0.000000e+00,4.423623e+07
new_cases_smoothed,408929.0,8.041026e+03,8.661611e+04,0.00,0.000,12.00,3.132900e+02,6.319461e+06
total_deaths,411804.0,8.125957e+04,4.411901e+05,0.00,43.000,799.00,9.574000e+03,7.057132e+06
new_deaths,410608.0,7.185214e+01,1.368323e+03,0.00,0.000,0.00,0.000000e+00,1.037190e+05
...,...,...,...,...,...,...,...,...
population,429435.0,1.520336e+08,6.975408e+08,47.00,523798.000,6336393.00,3.296952e+07,7.975105e+09
excess_mortality_cumulative_absolute,13411.0,5.604765e+04,1.568691e+05,-37726.10,176.500,6815.20,3.912804e+04,1.349776e+06
excess_mortality_cumulative,13411.0,9.766431e+00,1.204066e+01,-44.23,2.060,8.13,1.516000e+01,7.808000e+01
excess_mortality,13411.0,1.092535e+01,2.456071e+01,-95.92,-1.500,5.66,1.557500e+01,3.782200e+02


In [ ]:
pd.set_option('display.max_rows', None)
df.columns.tolist()

['iso_code',
 'continent',
 'location',
 'date',
 'total_cases',
 'new_cases',
 'new_cases_smoothed',
 'total_deaths',
 'new_deaths',
 'new_deaths_smoothed',
 'total_cases_per_million',
 'new_cases_per_million',
 'new_cases_smoothed_per_million',
 'total_deaths_per_million',
 'new_deaths_per_million',
 'new_deaths_smoothed_per_million',
 'reproduction_rate',
 'icu_patients',
 'icu_patients_per_million',
 'hosp_patients',
 'hosp_patients_per_million',
 'weekly_icu_admissions',
 'weekly_icu_admissions_per_million',
 'weekly_hosp_admissions',
 'weekly_hosp_admissions_per_million',
 'total_tests',
 'new_tests',
 'total_tests_per_thousand',
 'new_tests_per_thousand',
 'new_tests_smoothed',
 'new_tests_smoothed_per_thousand',
 'positive_rate',
 'tests_per_case',
 'tests_units',
 'total_vaccinations',
 'people_vaccinated',
 'people_fully_vaccinated',
 'total_boosters',
 'new_vaccinations',
 'new_vaccinations_smoothed',
 'total_vaccinations_per_hundred',
 'people_vaccinated_per_hundred',
 'peo

In [4]:
df.duplicated().sum()

np.int64(0)

MN 50 L 100 3MLT DROP 

W MN 30 L 50 3MLT ML IMPUTE BL EL RF

W MN 15 L 30 KNN 

 W EL BA2Y B MEDIAN AW MODE 3LA 7SAB EL TYPE BTA3T EL COLUMNS


 

In [5]:
pd.set_option('display.max_rows', None)
pct = df.isnull().mean().sort_values(ascending=False) * 100
pct

weekly_icu_admissions                         97.440125
weekly_icu_admissions_per_million             97.440125
excess_mortality                              96.877059
excess_mortality_cumulative_absolute          96.877059
excess_mortality_cumulative                   96.877059
excess_mortality_cumulative_per_million       96.877059
weekly_hosp_admissions                        94.295528
weekly_hosp_admissions_per_million            94.295528
icu_patients_per_million                      90.891287
icu_patients                                  90.891287
hosp_patients_per_million                     90.532677
hosp_patients                                 90.532677
total_boosters_per_hundred                    87.518484
total_boosters                                87.518484
new_vaccinations                              83.473401
new_tests                                     82.441347
new_tests_per_thousand                        82.441347
people_fully_vaccinated_per_hundred           81

In [6]:
drop_cols = [
    'weekly_icu_admissions',
    'weekly_icu_admissions_per_million',
    'excess_mortality',
    'excess_mortality_cumulative_absolute',
    'excess_mortality_cumulative',
    'excess_mortality_cumulative_per_million',
    'weekly_hosp_admissions',
    'weekly_hosp_admissions_per_million',
    'icu_patients_per_million',
    'icu_patients',
    'hosp_patients_per_million',
    'hosp_patients',
    'total_boosters_per_hundred',
    'total_boosters',
    'new_vaccinations',
    'new_tests',
    'new_tests_per_thousand',
    'people_fully_vaccinated_per_hundred',
    'people_fully_vaccinated',
    'total_tests',
    'total_tests_per_thousand',
    'people_vaccinated_per_hundred',
    'people_vaccinated',
    'total_vaccinations',
    'total_vaccinations_per_hundred',
    'tests_per_case',
    'positive_rate',
    'new_tests_smoothed_per_thousand',
    'new_tests_smoothed',
    'tests_units',
    'handwashing_facilities',
    'reproduction_rate',
    'new_people_vaccinated_smoothed',
    'new_people_vaccinated_smoothed_per_hundred',
    'new_vaccinations_smoothed_per_million',
    'new_vaccinations_smoothed',
    'stringency_index',
    'extreme_poverty'
]

df = df.drop(columns=drop_cols)


In [7]:
pct = df.isnull().mean().sort_values(ascending=False) * 100
pct


male_smokers                       43.223771
female_smokers                     42.444142
hospital_beds_per_thousand         32.308964
human_development_index            25.686774
aged_65_older                      24.722018
gdp_per_capita                     23.552575
cardiovasc_death_rate              23.419144
aged_70_older                      22.848627
median_age                         22.068998
diabetes_prevalence                19.449742
population_density                 16.054350
life_expectancy                     9.113370
continent                           6.176721
new_cases_smoothed                  4.775111
new_cases_smoothed_per_million      4.775111
new_deaths_smoothed_per_million     4.670555
new_deaths_smoothed                 4.670555
new_cases                           4.488689
new_cases_per_million               4.488689
new_deaths                          4.384133
new_deaths_per_million              4.384133
total_cases                         4.105627
total_deat

In [8]:
cols_to_impute = [
    'male_smokers',
    'female_smokers',
    'hospital_beds_per_thousand'
]
df_impute = df.copy()
rf_imputer = IterativeImputer(
    estimator=RandomForestRegressor(
        n_estimators=50,
        random_state=42,
        n_jobs=-1
    ),
    max_iter=10,
    random_state=42
)

df_impute[cols_to_impute] = rf_imputer.fit_transform(df_impute[cols_to_impute])

df = df_impute

c:\Users\modr3\anaconda3\Lib\site-packages\sklearn\impute\_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [9]:
pct = df.isnull().mean().sort_values(ascending=False) * 100
pct

human_development_index            25.686774
aged_65_older                      24.722018
gdp_per_capita                     23.552575
cardiovasc_death_rate              23.419144
aged_70_older                      22.848627
median_age                         22.068998
diabetes_prevalence                19.449742
population_density                 16.054350
life_expectancy                     9.113370
continent                           6.176721
new_cases_smoothed_per_million      4.775111
new_cases_smoothed                  4.775111
new_deaths_smoothed                 4.670555
new_deaths_smoothed_per_million     4.670555
new_cases                           4.488689
new_cases_per_million               4.488689
new_deaths_per_million              4.384133
new_deaths                          4.384133
total_deaths                        4.105627
total_cases                         4.105627
total_cases_per_million             4.105627
total_deaths_per_million            4.105627
location  

In [10]:
cols_to_impute = [
    'human_development_index',
    'aged_65_older',
    'gdp_per_capita',
    'cardiovasc_death_rate',
    'aged_70_older',
    'median_age',
    'diabetes_prevalence',
    'population_density'
]

scaler = StandardScaler()
scaled_data = scaler.fit_transform(df[cols_to_impute])

knn_imputer = KNNImputer(
    n_neighbors=3,
    weights="distance"  
)

imputed_data = knn_imputer.fit_transform(scaled_data)

df_imputed = pd.DataFrame(
    scaler.inverse_transform(imputed_data),
    columns=cols_to_impute
)

df[cols_to_impute] = df_imputed[cols_to_impute]


In [11]:
pct = df.isnull().mean().sort_values(ascending=False) * 100
pct

life_expectancy                    9.113370
continent                          6.176721
new_cases_smoothed_per_million     4.775111
new_cases_smoothed                 4.775111
new_deaths_smoothed                4.670555
new_deaths_smoothed_per_million    4.670555
new_cases                          4.488689
new_cases_per_million              4.488689
new_deaths                         4.384133
new_deaths_per_million             4.384133
total_cases_per_million            4.105627
total_deaths                       4.105627
total_cases                        4.105627
total_deaths_per_million           4.105627
location                           0.000000
date                               0.000000
iso_code                           0.000000
median_age                         0.000000
aged_65_older                      0.000000
aged_70_older                      0.000000
population_density                 0.000000
gdp_per_capita                     0.000000
cardiovasc_death_rate           

In [12]:
median_cols = [
    'life_expectancy',
    'new_cases_smoothed_per_million',
    'new_cases_smoothed',
    'new_deaths_smoothed',
    'new_deaths_smoothed_per_million',
    'new_cases',
    'new_cases_per_million',
    'new_deaths_per_million',
    'new_deaths',
    'total_deaths',
    'total_cases',
    'total_cases_per_million',
    'total_deaths_per_million'
]

mode_cols = ['continent']

for col in median_cols:
    df[col] = df[col].fillna(df[col].median())

for col in mode_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [13]:
pct = df.isnull().mean().sort_values(ascending=False) * 100
pct

iso_code                           0.0
continent                          0.0
location                           0.0
date                               0.0
total_cases                        0.0
new_cases                          0.0
new_cases_smoothed                 0.0
total_deaths                       0.0
new_deaths                         0.0
new_deaths_smoothed                0.0
total_cases_per_million            0.0
new_cases_per_million              0.0
new_cases_smoothed_per_million     0.0
total_deaths_per_million           0.0
new_deaths_per_million             0.0
new_deaths_smoothed_per_million    0.0
population_density                 0.0
median_age                         0.0
aged_65_older                      0.0
aged_70_older                      0.0
gdp_per_capita                     0.0
cardiovasc_death_rate              0.0
diabetes_prevalence                0.0
female_smokers                     0.0
male_smokers                       0.0
hospital_beds_per_thousan

In [17]:
def remove_outliers_iqr(df, cols):
    df_clean = df.copy()
    mask = pd.Series(True, index=df.index)

    all_outliers = []

    for col in cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        col_mask = (df[col] >= lower) & (df[col] <= upper)

        outliers = df[~col_mask]
        print(f"\n🔴 Outliers in {col}: {len(outliers)}")

        all_outliers.append(outliers)

        mask &= col_mask

    return df[mask], pd.concat(all_outliers)


df_clean, outliers_df = remove_outliers_iqr(df, cols_to_impute)


🔴 Outliers in human_development_index: 0

🔴 Outliers in aged_65_older: 0

🔴 Outliers in gdp_per_capita: 20088

🔴 Outliers in cardiovasc_death_rate: 21763

🔴 Outliers in aged_70_older: 1674

🔴 Outliers in median_age: 0

🔴 Outliers in diabetes_prevalence: 5022

🔴 Outliers in population_density: 0


In [18]:
df.to_csv('data.csv', index=False)   